<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/tapvidmv/review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pick the TAPVid-MV evaluation set

`shortlist.ipynb` narrows every episode down to a pool it can defend from
the metrics alone. What the metrics cannot see is whether the manipulation is
*interesting*, whether two candidates are doing the same thing anyway, and —
the one that matters most — whether the ground truth is actually **right**.

The quality cuts are computed from the tracker's own residuals, so an episode
whose tracks are confidently wrong scores well. Only looking at them catches it.

So each candidate plays as a **three-view clip with the ground-truth tracks drawn
on it**. A dot that slides across a surface, stays filled while the arm passes in
front of it, or drifts off the object it started on is a reject no threshold
would have found.

**How to use**

1. Run top to bottom. The prefetch cell renders clips in the background and
   caches them; the picker never waits for more than the next few.
2. **Keep** takes the episode, **Skip** drops it, **Back** undoes the last call.
3. The last cell writes `episodes_eval50.txt`, which `run_export.sh` reads.

---
## 0. Environment

Reads the mounted bucket on the pipeline machine, or pulls each episode's videos
and tracks into Colab as the picker reaches them.

In [ ]:
import os
import subprocess
import sys

if "google.colab" in sys.modules:
  subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "https://github.com/yangyi02/droid.git", "/content/droid"],
    check=True,
  )
  os.chdir("/content/droid/tapvidmv")
  subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ml_collections", "mediapy"], check=True)
  from google.colab import auth, output
  auth.authenticate_user()
  output.enable_custom_widget_manager()

HERE = os.getcwd()
REPO = os.path.dirname(HERE)
sys.path[:0] = [REPO, HERE]

import csv
import functools
import threading
from concurrent.futures import ThreadPoolExecutor

import cv2
import mediapy
import numpy as np
from IPython.display import display
import ipywidgets as widgets

import viz
from config import get_config
from shortlist import CUTS, cut_value

config = get_config()

In [ ]:
N_POOL = 100          # which pool to read: episodes_eval<N_POOL>.txt
N_TARGET = 50         # how many to keep

CLIP_FRAMES = 48      # frames per preview clip, resampled across the episode
CLIP_WIDTH = 440      # per-view width in the three-view strip
CLIP_TRACKS = 28      # how many tracks to draw
CLIP_TRAIL = 10       # frames of trail behind each dot
PREFETCH = 6          # clips rendered ahead of the one on screen

CANDIDATES = os.path.join(HERE, f"episodes_eval{N_POOL}_details.csv")
OUT_LIST = os.path.join(HERE, f"episodes_eval{N_TARGET}.txt")
CACHE = os.path.join(HERE, "previews")
GCS = "gs://dm-tapnet/tmp/droid"

os.makedirs(CACHE, exist_ok=True)
with open(CANDIDATES) as f:
  rows = list(csv.DictReader(f))

print(f"{len(rows)} candidates from {os.path.basename(CANDIDATES)}  ->  keep {N_TARGET}")

### 0.1 Where the episode data comes from

On the pipeline machine both roots are gcsfuse mounts and nothing is copied. In
Colab each episode is pulled on demand — the three `video_left.mp4` files and the
tracks, about 20 MB, fetched as the picker walks the pool rather than 2 GB up
front.

In [ ]:
def _pull(episode_id, stage, patterns):
  """Fetch one episode's files for a stage into the Colab cache, once."""
  local = os.path.join("/content/episodes", stage, episode_id)
  if os.path.isdir(local):
    return local
  os.makedirs(local, exist_ok=True)
  subprocess.run(
    ["gsutil", "-m", "cp", "-r"] + [f"{GCS}/{stage}/{episode_id}/{p}" for p in patterns] + [local],
    check=True,
    capture_output=True,
  )
  return local


@functools.lru_cache(maxsize=None)
def episode_roots(episode_id):
  """(depth_root, tracks_root) that hold this episode, mounted or pulled."""
  mounted_depth = os.path.expanduser(config.paths.depth)
  if os.path.isdir(os.path.join(mounted_depth, episode_id)):
    return mounted_depth, os.path.expanduser(config.paths.tracks)

  # gsutil drops the episode directory itself, so pull into <cache>/<stage>/<id>/..
  # and hand back the parent, keeping the same <root>/<episode_id>/.. shape.
  depth = _pull(episode_id, "depth", ["*/video_left.mp4"])
  tracks = _pull(episode_id, "tracks", ["*", ])
  return os.path.dirname(depth), os.path.dirname(tracks)

### 0.2 The clip

One row, three views, the same 3D points in the same colours in each. **Filled =
that view calls the point visible, hollow = occluded**, with a short trail behind
it.

Camera order comes from the tracks directory, because that is the order
`load_track_data` stacks `uv` and `vis` in — reading the depth directory
separately would risk pairing a view's frames with another view's points.

In [ ]:
import core.io


def render_clip(episode_id):
  """The three-view preview with ground truth drawn on it, as RGB frames."""
  depth_root, tracks_root = episode_roots(episode_id)
  tracks = core.io.load_track_data(episode_id, tracks_root)
  uv, vis = tracks["uv"], tracks["vis"]

  episode_dir = os.path.join(tracks_root, episode_id)
  cams = sorted(c for c in os.listdir(episode_dir) if os.path.isdir(os.path.join(episode_dir, c)))

  ids = viz.pick_tracks(vis, CLIP_TRACKS, require_views=2)
  colors = viz.track_colors(ids)

  n_frames = uv.shape[1]
  steps = np.unique(np.linspace(0, n_frames - 1, min(CLIP_FRAMES, n_frames)).astype(int))

  decoded, scales = [], []
  for view, cam in enumerate(cams):
    path = os.path.join(depth_root, episode_id, cam, "video_left.mp4")
    frames_by_step = viz.read_frames(path, steps) if os.path.exists(path) else {}
    # Resize first and scale the pixel coordinates with it. Drawing at full resolution and
    # letterboxing afterwards shrinks the dots and the label along with the image, which is
    # what makes a 4-pixel marker invisible in a 400-pixel panel.
    sample = next(iter(frames_by_step.values()), None)
    scale = CLIP_WIDTH / sample.shape[1] if sample is not None else 1.0
    decoded.append({
      step: cv2.resize(image, (CLIP_WIDTH, max(1, round(image.shape[0] * scale))), interpolation=cv2.INTER_AREA)
      for step, image in frames_by_step.items()
    })
    scales.append(scale)

  frames = []
  for step in steps:
    panels = []
    for view, cam in enumerate(cams):
      image = decoded[view].get(int(step))
      if image is None:
        continue
      trail_from = max(0, step - CLIP_TRAIL)
      canvas = viz.draw_trails(
        np.ascontiguousarray(image.copy()),
        uv[view, trail_from : step + 1][:, ids] * scales[view],
        colors,
        valid=vis[view, trail_from : step + 1][:, ids],
      )
      canvas = viz.draw_points(
        canvas, uv[view, step][ids] * scales[view], visible=vis[view, step][ids], colors=colors
      )
      panels.append(viz.label_panel(canvas, f"view {view} [{cam}]"))
    if panels:
      frames.append(viz.montage(panels, columns=len(panels), cell_width=CLIP_WIDTH))
  return np.stack(frames)


def clip_path(episode_id):
  return os.path.join(CACHE, episode_id.replace("+", "_") + ".mp4")


def ensure_clip(episode_id):
  """Render and cache one clip; returns the mp4 path, or None if it could not be built."""
  path = clip_path(episode_id)
  if os.path.exists(path):
    return path
  try:
    mediapy.write_video(path, render_clip(episode_id), fps=12)
  except Exception as failure:
    print(f"  {episode_id}: {type(failure).__name__}: {failure}")
    return None
  return path

### 0.3 Prefetch

Rendering decodes three videos, so it is the slow part. A background pool stays
`PREFETCH` clips ahead of the picker; episodes that fail to render are dropped
from the queue rather than stopping the run.

In [ ]:
pool = ThreadPoolExecutor(max_workers=3)
_futures = {}
_lock = threading.Lock()


def prefetch(index):
  """Keep the next PREFETCH clips rendering in the background."""
  with _lock:
    for row in rows[index : index + PREFETCH]:
      episode_id = row["episode_id"]
      if episode_id not in _futures:
        _futures[episode_id] = pool.submit(ensure_clip, episode_id)


def clip_bytes(episode_id):
  with _lock:
    future = _futures.get(episode_id)
  path = future.result() if future is not None else ensure_clip(episode_id)
  return open(path, "rb").read() if path else None


prefetch(0)
print(f"prefetching the first {PREFETCH} clips into {CACHE}")

---
## 1. Pick

**Keep** takes it, **Skip** drops it, **Back** undoes the last decision. The
counter stops you at `N_TARGET`. Nothing is written until the last cell.

The numbers beside each clip are the ones that let it through, reduced the way
the cuts read them — the worst camera for a ceiling, the weakest for a floor.

In [ ]:
def _f(row, key, fmt="{:.3f}"):
  try:
    return fmt.format(float(row[key]))
  except (KeyError, TypeError, ValueError):
    return "-"


def _worst(row, column, fmt="{:.1f}"):
  """The selector's own worst-camera reduction, so the picker judges on the number that cut."""
  value = cut_value(row, column, CUTS[column][0])
  return "-" if not np.isfinite(value) else fmt.format(value)


def summarise(row):
  episode_id = row["episode_id"]
  return (
    f"<b>{episode_id}</b><br>"
    f"scene {episode_id.split('+')[1]} &middot; site {row.get('site', '?')} &middot; "
    f"{_f(row, 'n_frames', '{:.0f}')} frames &middot; {_f(row, 'n_cameras', '{:.0f}')} cameras<br>"
    f"ee_travel <b>{_f(row, 'ee_travel_m')}</b> m &middot; "
    f"joint_range {_f(row, 'joint_range_max_rad')} rad &middot; gripper {_f(row, 'gripper_range')}<br>"
    f"cross_view <b>{_worst(row, 'cross_view_px')}</b> px &middot; "
    f"wrist {_worst(row, 'cross_view_wrist_px')} px &middot; "
    f"depth_residual {_worst(row, 'depth_residual_static_mm')} mm &middot; "
    f"chamfer {_worst(row, 'chamfer', '{:.3f}')}"
  )


state = {"index": 0, "kept": [], "history": []}

video = widgets.Video(format="mp4", autoplay=True, loop=True, width=1220)
info = widgets.HTML()
progress = widgets.HTML()
keep_button = widgets.Button(description="Keep", button_style="success", icon="check")
skip_button = widgets.Button(description="Skip", button_style="danger", icon="times")
back_button = widgets.Button(description="Back", icon="undo")


def show():
  index = state["index"]
  progress.value = (
    f"<b>{len(state['kept'])}</b> / {N_TARGET} kept &nbsp;&middot;&nbsp; "
    f"candidate <b>{min(index + 1, len(rows))}</b> / {len(rows)}"
  )
  if index >= len(rows) or len(state["kept"]) >= N_TARGET:
    info.value = "<b>Done.</b> Run the next cell to write the list."
    video.layout.display = "none"
    for button in (keep_button, skip_button, back_button):
      button.disabled = button is not back_button
    return

  row = rows[index]
  info.value = summarise(row)
  prefetch(index)
  payload = clip_bytes(row["episode_id"])
  if payload is None:
    info.value += "<br><i>clip failed to render - skipping</i>"
    state["index"] += 1
    show()
    return
  video.layout.display = ""
  video.value = payload


def decide(keep):
  index = state["index"]
  if index >= len(rows):
    return
  state["history"].append((index, keep))
  if keep:
    state["kept"].append(rows[index]["episode_id"])
  state["index"] = index + 1
  show()


def undo(_):
  if not state["history"]:
    return
  index, keep = state["history"].pop()
  if keep and state["kept"]:
    state["kept"].pop()
  state["index"] = index
  for button in (keep_button, skip_button):
    button.disabled = False
  show()


keep_button.on_click(lambda _: decide(True))
skip_button.on_click(lambda _: decide(False))
back_button.on_click(undo)

display(
  widgets.VBox([
    progress,
    video,
    info,
    widgets.HBox([keep_button, skip_button, back_button]),
  ])
)
show()

---
## 2. Write the evaluation set

Writes `episodes_eval50.txt` — the list `run_export.sh` reads.

Next: `bash tapvidmv/run_export.sh`

In [ ]:
kept = state["kept"]
scenes = {episode_id.split("+")[1] for episode_id in kept}
sites = {episode_id.split("+")[0] for episode_id in kept}

print(f"{len(kept)} episodes over {len(scenes)} scenes and {len(sites)} sites")

if not kept:
  print("Nothing kept - not writing.")
else:
  with open(OUT_LIST, "w") as f:
    f.writelines(episode_id + "\n" for episode_id in sorted(kept))
  print(f"wrote {OUT_LIST}")
  print("next:  bash tapvidmv/run_export.sh")